In [14]:
import os
import geopandas as gpd
import pandas as pd

import geowrangler.grids as grids
import geowrangler.vector_zonal_stats as vzs
import geowrangler.area_zonal_stats as azs

In [3]:
data_path = "../../data"

In [4]:
# Project to a metric CRS (e.g., EPSG:32651 for PH) 
# This is necessary for accurate area and distance calculations
aoi = gpd.read_file(
    os.path.join(data_path,"ph_adm3_municities/PH_Adm3_MuniCities.shp.shp")
).to_crs(epsg=32651)
aoi = aoi[aoi["geo_level"] == "City"]
aoi.head()

,adm1_psgc,adm2_psgc,adm3_psgc,adm3_en,geo_level,len_crs,area_crs,len_km,area_km2,geometry
4,100000000,102800000,102805000,City of Batac,City,66661,158252391,66,158.0,"POLYGON ((247341.309 2003933.537, 247293.327 2..."
11,100000000,102800000,102812000,City of Laoag,City,53964,110146974,53,110.0,"POLYGON ((248393.247 2016552.78, 248424.831 20..."
28,100000000,102900000,102906000,City of Candon,City,62247,77652664,62,77.0,"POLYGON ((230319.064 1907309.065, 230338.289 1..."
56,100000000,102900000,102934000,City of Vigan,City,25067,24485368,25,24.0,"POLYGON ((221597.447 1945779.016, 221718.087 1..."
70,100000000,103300000,103314000,City of San Fernando,City,54233,99006121,54,99.0,"POLYGON ((225191.401 1841887.86, 225339.01 184..."


In [5]:
def load_osm_gdfs(
    osm_file_path_dict: dict,
    aoi_crs,
) -> dict:
    return {
        key: gpd.read_file(val).to_crs(aoi_crs)
        for key, val in osm_file_path_dict.items()
    }

In [8]:
def calculate_osm_metrics(osm_path: str, year: int, osm_files: dict, aoi: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    # Filter criteria (Lowercase is required for OSM)
    osm_fclass = {
        "pois": ["school", "bank", "supermarket", "mall", "hospital"],
        "small_waterways": ["stream", "canal", "drain"],
        "large_waterways": ["river"],
        "gray_zones": ["residential", "industrial", "commercial", "retail"],
        "green_zones": ["forest", "grass", "park", "meadow", "scrub"]
    }

    # Load and Pre-process
    # Cast strings to 'object' to fix the StringDtype error
    pois = osm_files["pois"]
    pois["fclass"] = pois["fclass"].astype(object)

    waterways = osm_files["waterways"]
    waterways["fclass"] = waterways["fclass"].astype(object)
    waterways["length_m"] = waterways.geometry.length # Coverage Unit: METERS

    buildings = osm_files["buildings"]
    buildings["area_m2"] = buildings.geometry.area # Coverage Unit: SQ METERS

    landuse = osm_files["landuse"]
    landuse["fclass"] = landuse["fclass"].astype(object)
    landuse["area_m2"] = landuse.geometry.area # Coverage Unit: SQ METERS

    # Processing Chain (Updating aoi_results sequentially)
    aoi_results = aoi.copy()

    # A. Resilience Proxy (POI Counts)
    print("Getting zonal stats for pois")
    filtered_pois = pois[pois["fclass"].isin(osm_fclass["pois"])]
    aoi_results = vzs.create_zonal_stats(
        aoi_results,
        filtered_pois,
        overlap_method="intersects",
        aggregations=[{"func": ["count"], "output": "poi_count", "fillna": [True]}]
    )

    # B. Waterway Presence (Linear Meters)
    # Large Waterways
    print("Getting zonal stats for large waterways")
    aoi_results = vzs.create_zonal_stats(
        aoi_results,
        waterways[waterways["fclass"].isin(osm_fclass["large_waterways"])],
        aggregations=[{"column": "length_m", "func": ["sum"], "output": "large_water_m", "fillna": [True]}]
    )
    # Small Waterways
    print("Getting zonal stats for small waterways")
    aoi_results = vzs.create_zonal_stats(
        aoi_results,
        waterways[waterways["fclass"].isin(osm_fclass["small_waterways"])],
        aggregations=[{"column": "length_m", "func": ["sum"], "output": "small_water_m", "fillna": [True]}]
    )

    # C. Exposure & Runoff (Square Meters)
    # Building Area
    print("Getting zonal stats for buildings")
    aoi_results = vzs.create_zonal_stats(
        aoi_results,
        buildings,
        overlap_method="intersects", 
        aggregations=[{"column": "area_m2", "func": ["sum"], "output": "building_area_m2", "fillna": [True]}]
    )

    # Gray Areas (Accurate Runoff Potential)
    print("Getting area zonal stats for gray areas")
    aoi_results = azs.create_area_zonal_stats(
        aoi_results,
        landuse[landuse["fclass"].isin(osm_fclass["gray_zones"])],
        aggregations=[{"column": "area_m2", "func": ["sum"], "output": "gray_zones_area_m2", "fillna": [True]}]
    )
    
    # Green Areas (Accurate Infiltration Potential)
    print("Getting area zonal stats for green areas")
    aoi_results = azs.create_area_zonal_stats(
        aoi_results,
        landuse[landuse["fclass"].isin(osm_fclass["green_zones"])],
        aggregations=[{"column": "area_m2", "func": ["sum"], "output": "green_zones_area_m2", "fillna": [True]}]
    )
    
    aoi_results["year"] = year
    return aoi_results.fillna(0)

In [9]:
del osm_files

In [10]:
aoi_metrics_results = []

for year in range(2021, 2026):
    print(f"Processing {year}")
    
    osm_folder = f"openstreetmap/philippines-{year % 1000}0101-free.shp/"
    osm_path = os.path.join(data_path, osm_folder)
    osm_file_path_dict = {
        "waterways": os.path.join(osm_path, "gis_osm_waterways_free_1.shp"),
        "buildings": os.path.join(osm_path, "gis_osm_buildings_a_free_1.shp"),
        "pois": os.path.join(osm_path, "gis_osm_pois_free_1.shp"),
        "landuse": os.path.join(osm_path, "gis_osm_landuse_a_free_1.shp"),
    }
    
    print(f"Loading GDFs")
    osm_files = load_osm_gdfs(osm_file_path_dict, aoi.crs)
    print(f"Calculating OSM metrics")
    aoi_results = calculate_osm_metrics(osm_path, year, osm_files, aoi)
    aoi_metrics_results.append(aoi_results)

    # cleanup gdfs to free memory
    del osm_files

Processing 2022
Loading GDFs
Calculating OSM metrics
Getting zonal stats for pois
Getting zonal stats for large waterways
Getting zonal stats for small waterways
Getting zonal stats for buildings
Getting area zonal stats for gray areas
Getting area zonal stats for green areas
Processing 2023
Loading GDFs
Calculating OSM metrics
Getting zonal stats for pois
Getting zonal stats for large waterways
Getting zonal stats for small waterways
Getting zonal stats for buildings
Getting area zonal stats for gray areas
Getting area zonal stats for green areas
Processing 2024
Loading GDFs
Calculating OSM metrics
Getting zonal stats for pois
Getting zonal stats for large waterways
Getting zonal stats for small waterways
Getting zonal stats for buildings
Getting area zonal stats for gray areas
Getting area zonal stats for green areas


In [11]:
#

Processing 2025
Loading GDFs
Calculating OSM metrics
Getting zonal stats for pois
Getting zonal stats for large waterways
Getting zonal stats for small waterways
Getting zonal stats for buildings
Getting area zonal stats for gray areas
Getting area zonal stats for green areas


In [19]:
osm_metrics_cols = [
    "city_name",
    "psgc",
    "year",
    "poi_count",
    "large_water_m",
    "small_water_m",
    "building_area_m2",
    "gray_zones_area_m2",
    "green_zones_area_m2",
]

rename_cols = {"adm3_en": "city_name", "adm3_psgc": "psgc"}

ph_osm_metrics_df = pd.DataFrame(columns=osm_metrics_cols)

for ph_osm_df in aoi_metrics_results:
    _ph_osm_df = ph_osm_df.copy()
    _ph_osm_df.rename(columns={"adm3_en": "city_name", "adm3_psgc": "psgc"}, inplace=True)
    ph_osm_metrics_df = pd.concat(
        [ph_osm_metrics_df , _ph_osm_df[osm_metrics_cols]],
        axis=0,
        ignore_index=True
    )

ph_osm_metrics_df 

,city_name,psgc,year,poi_count,large_water_m,small_water_m,building_area_m2,gray_zones_area_m2,green_zones_area_m2
0,City of Batac,102805000,2022,11.0,34518.30456,10091.147335,1458817.042814,51077.935611,6623.38008
1,City of Laoag,102812000,2022,24.0,70043.827471,7755.936046,3181258.117859,1427905.757674,911009.351456
2,City of Candon,102906000,2022,12.0,46861.84554,6097.789628,1591184.891838,371598.218724,72709.547765
3,City of Vigan,102934000,2022,23.0,20236.390713,1115.210599,1689781.688349,79766.835676,11646.161939
4,City of San Fernando,103314000,2022,47.0,17987.10405,29439.919307,2021883.412402,389387.058478,21974.975861
...,...,...,...,...,...,...,...,...,...
591,City of Calapan,1705205000,2025,32.0,137063.477722,32255.799245,995685.269952,2925976.602136,563194.329973
592,City of Puerto Princesa,1731500000,2025,39.0,518006.293132,131744.839757,5897382.574775,14055158.436186,653369365.022192
593,City of Lamitan,1900702000,2025,3.0,44460.59846,3323.283848,200384.410367,1764670.758231,29245.665994
594,City of Marawi,1903617000,2025,26.0,15188.292204,18930.609236,2369239.784338,9727853.946625,7124492.979279


In [24]:
# Validation: All cities have some OSM value
ph_osm_metrics_df[["city_name", "psgc", "year"]].groupby("year").agg("nunique")

,city_name,psgc
year,,
2022,145,149
2023,145,149
2024,145,149
2025,145,149


In [26]:
output_path = os.path.join(data_path, "urban_osm_metrics.csv")
ph_osm_metrics_df.to_csv(output_path, index=False)